<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures%5C5a_XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XAI (ExplainableAI)

Sources:   
* Books/Manuscripts
  * [[IML] Interpretable Machine Learning: A Guide for Making Black Box Models Explainable, by Christoph Molnar](https://christophm.github.io/interpretable-ml-book/)  
  * Rami Ibrahim and M. Omair Shafiq, Explainable Convolutional Neural Networks: A Taxonomy, Review, and Future Directions, ACM Computing Surveys, Vol. 55, No. 10, Article 206, 2023 https://dl.acm.org/doi/pdf/10.1145/3563691
  * Ponce-Bobadilla AV, Schmitt V, Maier CS, Mensing S, Stodtmann S. Practical guide to SHAP analysis: Explaining supervised machine learning model predictions in drug development. Clin Transl Sci. 2024 Nov;17(11):e70056. doi: 10.1111/cts.70056. PMID: 39463176; PMCID: PMC11513550.

* Python Packages  
[SHAP](https://github.com/shap/shap#deep-learning-example-with-gradientexplainer-tensorflowkeraspytorch-models) [API](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html)  
[CAPTUM](https://captum.ai/docs/attribution_algorithms)  
[TF-Explain](https://pypi.org/project/tf-explain/) [GitHUB](https://github.com/sicara/tf-explain)  

---
---


# I. Interpretability and Explainability
What is the differnce between **interpretability** and **explainability**?
They are typically used interchangeably though some differences maybe applicable in a context.
Interpretability is mapping a blackbox model  (such as deep learning) into an interpretable form.
Explainability is a stronger term requiring interpretability and additionally explainining the decisions made by the model, which is important for engineering applications where model outputs are used for informing decisions.

Methods used do not change.
We will refer to the general area as XAI

---
---

### Ia. **[XAI Taxonomy](https://christophm.github.io/interpretable-ml-book/images/taxonomy.jpg)**
Taxonomy is evolving as rapidly as the field is. Generally, techniques can be catergorized as below
* Intrinsic: could be by design e.g., linear regression, or decision trees; antehoc: RNN with Attention mechanisms).
* Post-hoc: Attached to explain the model **(most deep learning methods need this type of techniques**)
  * model-agnostic: applicable to a range of ML models
    * global: interpretation of model generally
    * local: interpretation of specific predictions
  * model-specific: applicable to specific type of ML models

Another categorization of mostly post-hoc methods specific to deep learning:
* feature attribution,
* layer attribution, or
* neuron attribution.
(There are other newly emerging techniques on concept attribution for CNN) (see [CAPTUM](https://captum.ai/docs/attribution_algorithms)

---
---

## Ib. Intrinsic method: LInear regression interpretation
<p align="center">
  <img src="https://christophm.github.io/interpretable-ml-book/limo_files/figure-html/fig-linear-weights-plot-1.png" width="35%" />
  <img src="https://christophm.github.io/interpretable-ml-book/limo_files/figure-html/fig-linear-effects-1.png" width="35%" />
</p>

Source: [IML](https://christophm.github.io/interpretable-ml-book/)  Weight = coefficient;   Effect = weight X feature value

<p align="center"><em>Figure 6.1: Linear regression estimates for bike rental data. Weights are displayed as points and the 95% confidence intervals as lines.  Figure 6.2: Effect plot for linear regression results for the bike rental data. The boxplots show the distribution of effects (= feature value times feature weight) across the data per feature. </em></p>

* To make the estimated weights more comparable, scale the features (zero mean and standard deviation of one) before fitting the linear model.
* The weights depend on the scale of the features, i.e., it will be different if you change the features unit, e.g., a person’s height in meter v. centimeter will hvae differnt weights. Though the weight will change, the actual effect will not.

---
---

# II. Post-hoc Local model-agnostic methods

Model agnostic
* Ceteris paribus (CP) plots
* Individual conditional expectation (ICE)
* LIME (Local Interpretable Model-Agnostic Explanations)(for classification problems)
* Anchor (for classification problems)
* Shapley values
* [SHAP (SHapley Additive exPlanation)](https://papers.nips.cc/paper_files/paper/2017/file/8a20a8621978632d76c43dfd28b67767-Paper.pdf)

Model-specific: Models with differntiable functions (deep learning)(most gradient-based)
* Integrated gradients
* DeepLift
* DeepLiftSHAP
* GradientSHAP
* Guided GradCAM
* Layer GradCAM

---


### IIa. **Ceteris paribus plots**
#### Core concept of CP: How does changing individual features change the predicted values?

 - Train a model as usual
 - Use the trained model to predict the response output when keeping everything at a selected baseline, and then vary each feature individually.

 Example: PLots below predict number of bike rentals.

 First plot: Show CP applied to SVM (Support Vector Machine) model. Dots on the first plot show the baseline values. The curves show CP for contnuous features, and bar plots show CP for discrete features. For example, keeping everything fixed, except varying temperature has a inverted u-shape shape (bike rentals initially increase as temp increases, and then decreases when temperatures get too high).

Second plot: CP for temperature when using different machine learning models. NOtice that the 'linear model' does not the capture the non-linear inverted U as SVM.
<table align="center">
  <tr>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://christophm.github.io/interpretable-ml-book/ceteris-paribus_files/figure-html/fig-whatif-bike-1.png" width="400px" />
        <figcaption>Ceteris Paribus Profile for Bike Rentals: <br> How changing individual features changes the <br>predicted number of bike rentals.</figcaption>
      </figure>
    </td>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://christophm.github.io/interpretable-ml-book/ceteris-paribus_files/figure-html/fig-cp-bike-models-1.png" width="400px" />
        <figcaption>Comparing Multiple Models with CP Profiles: <br>Ceteris paribus curves for the bike prediction task for different models:<br> linear model, random forest, SVM, and decision tree</figcaption>
      </figure>
    </td>
  </tr>
</table>

Source: [IML](https://christophm.github.io/interpretable-ml-book/)  
Dataset: [Bike rental prediction](https://christophm.github.io/interpretable-ml-book/data.html#bike-data)

Strengths
* Simple to do and interpret
* Complement attribution-based methods that do not show how sensitive the prediction function to local changes (like SHAP or LIME) by providing a complete picture to explaining individual predictions.
* Ceteris paribus plots are flexible building blocks.

Limitations
* Ceteris paribus plots only show us one feature change at a time.
* Interpretation suffers when features are correlated.



---

### IIb. **Individual Conditional Expectation (ICE)**


#### **ICE** are CP for every instance (sample): The values for each sample (each row in tabular dataset) are computed by keeping all other features the same.

Suppose we have 2 features, x1, x2 and N samples. For a plot of x2 against y, we will have N lines, each with x1 fixed at value of that sample, while x2 varies in its full range. Repeat for all features.

**Centered ICE**
It can be hard to tell whether the ICE curves differ between data points because they start at different predictions. A simple solution is to center the curves at a certain point in the feature and display only the difference in the prediction to this point

**Derivative ICE**
Instead of actual value, plot the partial derivatives. If there are no interaction terms, the derivatives will not change for every sample.

---

<table align="center">
  <tr>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://raw.githubusercontent.com/chaitragopalappa/MIE590-690D/main/images/ICE_quadratic%20function.png" width="400px" />
        <figcaption>ICE: y=x1^2 + x2^2.</figcaption>
      </figure>
    </td>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://raw.githubusercontent.com/chaitragopalappa/MIE590-690D/main/images/CenteredICE_quadratic.png" width="400px" />
        <figcaption>Centered ICE : y=x1^2 + x2^2</figcaption>
      </figure>
    </td>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://raw.githubusercontent.com/chaitragopalappa/MIE590-690D/main/images/derivativeICE_quadratic.png" width="400px" />
        <figcaption>Derivative ICE: y=x1^2 + x2^2</figcaption>
      </figure>
    </td>
  </tr>
</table>

-----

<table align="center">
  <tr>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://raw.githubusercontent.com/chaitragopalappa/MIE590-690D/main/images/ICE_interaction_term.png" width="400px" />
        <figcaption>ICE: y=x1^2 + x2^2 + x1 x2.</figcaption>
      </figure>
    </td>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://raw.githubusercontent.com/chaitragopalappa/MIE590-690D/main/images/CenteredICE_INteractionTerm.png" width="400px" />
        <figcaption>Centered ICE : y=x1^2 + x2^2 + x1 x2</figcaption>
      </figure>
    </td>
    <td style="text-align: center; padding: 8px;">
      <figure>
        <img src="https://raw.githubusercontent.com/chaitragopalappa/MIE590-690D/main/images/derivativeICE_interactionTerm.png" width="400px" />
        <figcaption>Derivative ICE: y=x1^2 + x2^2 + x1 x2</figcaption>
      </figure>
    </td>
  </tr>
</table>

---

### IIc.  **LIME (Local Interpretable Model-Agnostic Explanations) (works for tabular data, images, and text)**
#### Asks a different question than CP and ICE. For a given sample, why does the model predict the outcome it did? It does so by fitting a local interpretable surrogate. That is, it perturbs the inputs around that sample, predicts corresponding outputs, and fits an interpretable linear model to these perturbed outputs

Originally developed for interpreting blackbox classifiers: Ribeiro et.al., “Why Should I Trust You?” Explaining the Predictions of Any Classifier, ACM, 2016  https://arxiv.org/pdf/1602.04938 ; [Related Blog](https://www.oreilly.com/content/introduction-to-local-interpretable-model-agnostic-explanations-lime/) ; [Python files](https://github.com/marcotcr/lime)

Recent applications have expanded to regression problems as well: https://github.com/marcotcr/lime/tree/ce2db6f20f47c3330beb107bb17fd25840ca4606

LIME is an interpretable "local" surrogate model of a black-box model- the local surrogate could be linear, Lasso, decision tree etc. Works by generating a new dataset consisting of perturbed samples and the corresponding predictions of the black box model. On this new dataset, LIME then trains an interpretable model, which is weighted by the proximity of the sampled instances to the instance of interest. The learned model is "local" surrogate  (or local fidelity) not a global surrogate.
$$ explanation(x)=arg min_{g\in G} L(\hat{f},g,\pi_x)+\Omega(g)$$
* $g$ is the explanation model for instance $x$ (e.g., linear regression model)
  * $G$ is a class of linear models, such that, for some perturbed instance $z'$, we have  $g(z') = w_g · z'$
* $\hat{f}$  is the orginal prediction model (e.g., neural net)
* $\Omega$ is model complexity (defined by the user, e.g., prefer fewer features),
* $L$ is loss (e.g., mean squared error) between $g$ and $\hat{f}$ and
* $\pi_x$ is the proximity measure defining how large the neighborhood around instance $x$ is that we consider for the explanation.
  * LIME currently uses an exponential smoothing kernel to define the neighborhood.
  * $\pi_x(z) = exp(-D(x, z)^2 / \sigma^2)$ ;
  * $D$ is distance function; (e.g. cosine distance for text, L2 distance for tabular and images)
  * $\sigma$ is width; a small kernel width means that an instance must be very close to influence the local model; a larger kernel width means that instances that are farther away also influence the model.

Strengths:
* MOdel agnostic (actual black-box model could be any model)
* Works on tabular data, images, and text data
* Easy to use

Limitations
* Unstable if linear approximations are not sufficient even in a local neighborhood

---


**LIME Examples**

<p align="center">
  <img src="https://raw.githubusercontent.com/marcotcr/lime/master/doc/images/lime.png" width="50%" /> "
  
</p>
 Figure 3: "*Toy example to present intuition for LIME. The black-box model’s complex decision function f (unknown to LIME) is represented by the blue/pink background, which cannot be approximated well by a linear model. The bold red cross is the instance being explained. LIME samples instances, gets predictions using f , and weighs them by the proximity to the instance being explained (represented here by size). The dashed line is the learned explanation that is locally (but not globally) faithful.*"

 Source: Ribeiro et.al., “Why Should I Trust You?” Explaining the Predictions of Any Classifier, ACM, 2016  https://arxiv.org/pdf/1602.04938

 * [Code 1 - RandomForests Regression](https://marcotcr.github.io/lime/tutorials/Using%2Blime%2Bfor%2Bregression.html)
  
```
Intercept 23.9047475063 # intercept of local surrogate
Prediction_local [ 22.32579479] # prediction from local surrogate
Prediction_Original: 23.1073 # Actual value -from original prediction model

# Range feature, Coefficients from local linear surrogate
[('6.99 < LSTAT <= 11.43', 1.7571320048618118),
 ('6.21 < RM <= 6.62', -1.5638211582388033),
 ('NOX > 0.62', -0.77384372989110417),
 ('19.10 < PTRATIO <= 20.20', -0.60756112694664299),
 ('2.08 < DIS <= 3.17', -0.39085870918058263)]
```
* [Code 2- Classification using RandomForests and XGBoost](https://marcotcr.github.io/lime/tutorials/Tutorial%20-%20continuous%20and%20categorical%20features.html) ;
  



 ---
 ---

### IId. **Anchor (for classfication problem)**
 #### Asks the same question as LIME. For a given sample, why did the model predict the outcome it did? It fits a local surrogate but puts a bound around where that surrogate holds, by putting an anchor (set of rules). The surrogate model applies to only samples that follow those set of rules.

Original article: Anchors: Ribeiro, M. T., Singh, S., & Guestrin, C. (2018). Anchors: High-Precision Model-Agnostic Explanations. Proceedings of the AAAI Conference on Artificial Intelligence, 32(1). https://doi.org/10.1609/aaai.v32i1.11491 ; [Python](https://github.com/marcotcr/anchor?tab=readme-ov-file)

Instead of using surrogate linear models used by LIME, this method uses easy-to-understand IF-THEN rules, called anchors. LIME results do not indicate how faithful they are, as LIME solely learns a linear decision boundary that best approximates the model given a perturbation space. Given the same perturbation space, the anchors approach constructs explanations whose coverage is adapted to the model’s behavior, and the approach clearly expresses their boundaries. Thus, they are faithful by design and state exactly for which instances they are valid. This property makes anchors particularly intuitive and easy to comprehend.

See vizual example comparing Anchor with LIME : https://christophm.github.io/interpretable-ml-book/anchors.html

---
Example: Titanic dataset: predict whether or not a passenger survived the Titanic disaster.

Anchor is used to explain why the model predicts 'survived' for this specifc person  
**Feature |     Value**  
Age |	20  
Sex |	female  
Class |	first  
Ticket price |	300$  
More attributes 	…  
Survived |	true  

And the corresponding anchors explanation is:
IF SEX = female AND Class = first THEN PREDICT Survived = true WITH PRECISION 97% AND COVERAGE 15%

---

**Mathematically**  
An **anchor** $A$ is a rule that “anchors” a prediction $\hat{f}(\mathbf{x})$ if, for all samples $\mathbf{z}$ that satisfy the same rule $A$, the model predicts the same outcome with high probability:

$$
\mathbb{E}_{\mathcal{D}_\mathbf{x}(\mathbf{z}\mid A)}[1_{\hat f(\mathbf{x}) = \hat f(\mathbf{z})}] \ge \tau
\quad \text{and} \quad A(\mathbf{x}) = 1
$$

where:  
- $\mathbf{x}$: instance being explained  
- $A$: the anchor rule (set of feature predicates)  
- $\hat{f}$: the black-box prediction model  
- $\mathcal{D}_\mathbf{x}(\mathbf{z}\mid A)$: distribution of perturbed samples satisfying $A$  
- $\tau$: desired precision threshold (0–1)


Anchors guarantee that the rule $A$ holds with **high precision** and **confidence**:

$$
\mathbb{P}( \mathrm{prec}(A) \ge \tau ) \ge 1 - \delta
\quad \text{where} \quad
\mathrm{prec}(A) = \mathbb{E}_{\mathcal{D}_\mathbf{x}(\mathbf{z}\mid A)}[1_{\hat f(\mathbf{x}) = \hat f(\mathbf{z})}]
$$

Here $\delta$ is a small value (e.g., 0.05) representing the acceptable uncertainty.

The **coverage** of an anchor $A$ is the expected fraction of instances for which the rule applies:

$$
\mathrm{cov}(A) = \mathbb{E}_{\mathcal{D}_{(\mathbf{x})}}[A(\mathbf{z})]
$$

---

**Findings Anchors**
Objective function: Find the anchor rule $A$ that maximizes coverage while satisfying the precision constraint:
$$
\underset{A : \mathbb{P}(\mathrm{prec}(A)\ge \tau)\ge1 - \delta}{\max} \; \mathrm{cov}(A)
$$
Anchors utilizes reinforcement learning techniques (specifically multi-armed bandits (MABs)) in combination with a graph search algorithm to reduce the number of model calls (and hence the required runtime) to a minimum while still being able to recover from local optima. To this end, neighbors, or perturbations, are created and evaluated for every instance that is being explained. Doing so allows the approach to disregard the black box’s structure and its internal parameters so that these can remain both unobserved and unaltered.

Strengths:
* Easy to interpret
* Anchors are Subsettable: trade-off between precision and coverage
* Works even when model is non-linear around nieghborhood of the instance being interpreted
* Model-agnostic
* Anchors shed light on the robustness of the original prediction model to changes in features
* It is efficient and be parallelized by using MABs that allow bacth sampling (e.g., BatchSAR)

Limitations
* Highly ocnfigurable : Needs hyperparameter tuning - beam width and precision threshold; Perturbation functions need to be explicity defined for each use case.
* Many scenarios requrie discretization, which in some cases maynot work.
* Computational complexity as with any perturbation based methods - multiple calls to prediction (though use of MAB reduces this to some extent)

**Code**: [Python Package](https://github.com/marcotcr/anchor?tab=readme-ov-file)

---
---

### IIe. **SHAP v. Shapley values**

Source: [Interpretable Machine Learning: A Guide for Making Black Box Models Explainable, by Christoph Molnar](https://christophm.github.io/interpretable-ml-book/shap.html)

*"SHAP (SHapley Additive exPlanations) by Lundberg and Lee (2017) is a method to explain individual predictions. SHAP is based on the game-theoretically optimal Shapley values. I recommend reading the chapter on Shapley values first.*

*To understand why SHAP is a thing and not just an extension of the Shapley values chapter, a bit of history: In 1953, Lloyd Shapley introduced the concept of Shapley values for game theory. Shapley values for explaining machine learning predictions were suggested for the first time by Štrumbelj and Kononenko (2011) and Štrumbelj and Kononenko (2014). However, they didn’t become so popular. A few years later, Lundberg and Lee (2017) proposed SHAP, which was basically a new way to estimate Shapley values for interpreting machine learning predictions, along with a theory connecting Shapley values with LIME and other post-hoc attribution methods, and a bit of additional theory on Shapley values.*

*You might say that SHAP is just a rebranding of Shapley values (which is true), but that would miss the fact that SHAP also marks a change in popularity and usage of Shapley values, and introduced new ways of estimating Shapley values and aggregating them in new ways. Also, SHAP brought Shapley values to text and image models."*

---

Before we go to SHAP for ML lets understand what Shapley values are.

### **Background on Shapley values**

The Shapley value was coined by Shapley (1953) for cooperative games. The Shapley value is a method (solution concept) for fairly distributing the total gains (or payouts) among a group of players who have collaborated. Players cooperate in a coalition and receive a certain profit from this cooperation. Shapley value is a method for assigning payouts to players depending on their contribution to the total payout.

Example: Suppose a company that delivers packages to three cities: A, B, and C. Each city is served by one truck:
• Truck 1 → serves City A
• Truck 2 → serves City B
• Truck 3 → serves City C

Each truck can make deliveries independently, but if they work together, they can share routes, reduce empty trips, and improve efficiency.

| **Participating Trucks** | **Profit ($ K)** |
|:--------------------------|:----------------:|
| None                     | 0                |
| A only                   | 10               |
| B only                   | 20               |
| C only                   | 15               |
| A + B                    | 40               |
| A + C                    | 35               |
| B + C                    | 45               |
| A + B + C                | 60               |

The total gains or payout from the coalition of A,B,C is $60. The Shapley value determines how to fairly allocate this to each player, by estimating how much the overall outcome changes when they join each possible coalition (combination of other players), and then averaging those changes.

In essence, it calculates each player's average marginal contribution across all possible coalitions. It is the only solution that satisfies four fundamental properties: efficiency, symmetry, additivity, and the dummy player (or null player) property, which are widely accepted as defining a fair distribution.

Shapley value for player $j$ is calculated as
$$\phi(j)​=\sum_{S⊆N∖{j}} \frac{​∣S∣!(M−∣S∣−1)!}{M!}[v(S∪{j})−v(S)]$$
- $M$ is number of players;
- $S$ is a coalition; $N$ is all possible coalition;
- $N ∖j $ is all possible coalition not containing player j;
- $v$ is value of the coalition.
- Thus, for player A, $S$ would be all colaitions excluding A, $S \in \{BC, B, C, \phi\}$

The correponding Shapley value for A =
$\phi(A)=  
(2!(3-2-1)!/3!)(v(B,C,A)-v(B,C)) +      
(1!(3-2-1)!/3!)( v(B,A)-v(B)) +   
(1!(3-2-1)!/3!)(v(C,A)-v(C)) +     
(0!(3-2-1)!/3!) (val(A)-val(null))     
= (60-45)2/6+(40-20)1/6+(35-15)1/6+(10-0)2/6 =15$  

---

**Intuitive**: Write possible coalitions in order of players joining coalition, there are 3! possible coalitions. Then take average of all marginal contribution of A
| Order | Sequence | Marginal contribution of player 1 | Marginal contribution of player 2 | Marginal contribution of player 3 |
|:------:|:----------|:----------------------------|:----------------------------|:----------------------------|
| 1 | A, B, C | A = 10 | B = 30 (40–10) | C = 20 (60–40) |
| 2 | A, C, B | A = 10 | C = 25 (35–10) | B = 25 (60–35) |
| 3 | B, A, C |B = 20 | A = 20 (40–20) |  C = 20 (60–40) |
| 4 | B, C, A | B = 20|  C = 25 (45–20) | A = 15 (60–45) |
| 5 | C, A, B |  C = 15 |A = 20 (35–15) | B = 25 (60–35) |
| 6 | C, B, A |  C = 15 | B = 30 (45–15) |A = 15 (60–45) |

$\phi(A)= (10 +10+20+15+20+15)/6$  
$\phi(B)= (30+25+20+20+25+30)/6$  
$\phi(C)= (20+25+20+25+15+15)/6$

---
**Shapley value Properties**
The Shapley value is the only attribution method that satisfies all the below properties, which are widely accepted as defining a fair distribution of the payout.

   * **Efficiency**: The players' contributions must add to the total payout.
   * **Symmetry**: The contributions of two players should be the same if they contribute equally to all possible coalitions.
   * **Dummy**: A feature that does not contribute should have a Shapley value of 0
   * **Additivity**: The sum of the Shapley values for two combined games is the same as the sum of the individual games' Shapley values.

   ---
   ---




#### **Shapley values for machine learning** (classfication and regression)

**It is applied to each sample prediction. It asks the following question: how much did each input feature contribute to that sample's prediction.**

**We can interpret this as a game: suppose every sample is a game. The payout for that game = sample prediction - average prediction across all samples; How much did each feature contribute to that payout?**  

Computes feature contributions for a specific prediction.

Shapley value for **feature** $j$ is calculated as
$$\phi(j)​=\sum_{S⊆N∖{j}} \frac{​∣S∣!(M−∣S∣−1)!}{M!}[v(S∪{j})−v(S)]$$
$M$ is number of features; $S$ is a coalition; $N$ is all possible coalition; $N ∖j $ is all possible coalition not containing feature j; $v$ is value of the coalition.

$$v_x (S) =\int \hat{f}(x_1, x_2, ...x_M)d\mathbb{P}_{X_c}-\mathbb{E}[\hat{f}(X)]$$
We actually perform multiple integrations for each feature that is not contained in $S$. Example: Suppose we have 4 features $X_1,X_2,X_3,X_4$ and we evaluate the prediction for the coalition consisting of feature values $x_1,x_3$ then :

$$v_x(S)=v_x(1,3) =\int _{\mathbb{R}} \int _{\mathbb{R}}  \hat{f}(x_1, X_2, x_3, X_4)d\mathbb{P}_{X_2X_4}-\mathbb{E}[\hat{f}(X)] $$

**Note**: With this representation, the sum of Shapely values for a specific prediction will be the difference in the predicted value and the expected value (this leads to the efficiency property of Shapley). To demonstrate this let's calculate Shapley values for a known function.

---
---






**Note**: The sum of Shapely values for a specific prediction would then be the difference in the predicted value for that sample and the expected value across all samples (this leads to the efficiency property of Shapley). To demonstrate this let's calculate Shapley values for a known function.

**Calculate Shapley values for a known function** $$\hat f(x) = a x_1 + b x_2$$

**Step 1: compute expected model prediction**
$\mathbb{E}[\hat f(X)] = a\,\mathbb{E}[X_1] + b\,\mathbb{E}[X_2] = a\mu_1 + b\mu_2.$  

---
**Step 1: compute values $v_x(S)$  for each coalition (each subset of features)**

$$v_x(S) = \int \hat f(x_S, X_C)\, d\mathbb{P}_{X_C} - \mathbb{E}[\hat f(X)],$$
where $C = N \setminus S$ are the unknown features being integrated out; and  $d\mathbb{P}_{X_C}$ indicates integration over the probability distribution of $X_c$

* $S = \varnothing$; No features are known.  We integrate over both \(X_1\) and \(X_2\):
$$
\begin{aligned}
v_x(\varnothing)
&= \int_{X_1 }  \int_{X_2} \big(a X_1 + b X_2\big) d\mathbb{P}_{X_1, X_2}
    - \big(a\mu_1 + b\mu_2\big)
&= a\mu_1 + b\mu_2 - a\mu_1 - b\mu_2
&= 0.
\end{aligned}
$$

* $S = \{1\}$; We know $x_1$, integrate over $X_2$:
$$
\begin{aligned}
v_x(\{1\})
&= \int \hat f(x_1, X_2)\, d\mathbb{P}_{X_2}
    - \mathbb{E}[\hat f(X)]
&= \int \big(a x_1 + b X_2\big) d\mathbb{P}_{X_2}
    - (a\mu_1 + b\mu_2)
&= a x_1 + b \mathbb{E}[X_2] - a\mu_1 - b\mu_2
&= a(x_1 - \mu_1).
\end{aligned}
$$

* $S = \{2\}$; We know $x_2$, integrate over $X_1$:
$$
\begin{aligned}
v_x(\{2\})
&= \int \hat f(X_1, x_2)\, d\mathbb{P}_{X_1}
    - \mathbb{E}[\hat f(X)]
&= \int \big(a X_1 + b x_2\big) d\mathbb{P}_{X_1}
    - (a\mu_1 + b\mu_2) \\[4pt]
&= b(x_2 - \mu_2).
\end{aligned}
$$
* $S = \{1,2\}$; We know both features; no integration needed:
$$
\begin{aligned}
v_x(\{1,2\})
&= \hat f(x_1, x_2) - \mathbb{E}[\hat f(X)]
&= (a x_1 + b x_2) - (a\mu_1 + b\mu_2)
&= a(x_1 - \mu_1) + b(x_2 - \mu_2).
\end{aligned}
$$

---

 **Step 2. Compute marginal contributions for each feature for all possible coalitions after excluding that feature**

For feature 1:
$$
\begin{aligned}
v_x(\{1\}) - v_x(\varnothing) &= a(x_1 - \mu_1), \\[4pt]
v_x(\{1,2\}) - v_x(\{2\}) &= a(x_1 - \mu_1).
\end{aligned}
$$

For feature 2:
$$
\begin{aligned}
v_x(\{2\}) - v_x(\varnothing) &= b(x_2 - \mu_2), \\[4pt]
v_x(\{1,2\}) - v_x(\{1\}) &= b(x_2 - \mu_2).
\end{aligned}
$$

---

**Step 3. Calculate Shapley value for each feature as the average of marginal contributions from all coalitions**

$$\phi_1 = a(x_1 - \mu_1), \quad
\phi_2 = b(x_2 - \mu_2)$$
**Note**: sum of Shapley values for a specific prediction is the difference in the predicted value and the expected value. **Note: This depicts  the efficiency property of Shapley values**
$$
\sum_{j=1}^{p} \phi_j(\hat{f})= \hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{X})]
$$

or

$$
\hat{f}(\mathbf{x}) = \sum_{j=1}^{p} \phi_j(\hat{f}) + \mathbb{E}[\hat{f}(\mathbf{X})]
$$



---
---


**Another way of interpretting Shapley in Linear Models**
### The part connecting Shapely to modern day SHAP (SHapley Additive exPlanation)
The linear model is given by $
\hat{f}(\mathbf{x}) = \beta_0 + \beta_1 x_1 + \ldots + \beta_p x_p
$

where **x** is the instance for which we want to compute the contributions.  
Each $x_j$ is a feature value, with $j = 1, \ldots, p$.  
The $\beta_j$ is the weight corresponding to feature $j$.


The contribution $\phi_j$ of the $j$-th feature to the prediction $\hat{f}(\mathbf{x})$ is the difference between the feature effect and the average effect:

$$
\phi_j(\hat{f}) = \beta_j x_j - \beta_j \mathbb{E}[X_j]
$$

where $\beta_j \mathbb{E}[X_j]$ is the mean effect estimate for feature $j$.  
  

Now we know how much each feature contributed to the prediction.  If we sum all the feature contributions for one instance, the result is the following:

$$
\sum_{j=1}^{p} \phi_j(\hat{f})
= \sum_{j=1}^{p} (\beta_j x_j - \beta_j \mathbb{E}[X_j])
$$

$$
\sum_{j=1}^{p} \phi_j(\hat{f})
=  \left( \beta_0 + \sum_{j=1}^{p} \beta_j x_j \right)
- \left( \beta_0 + \sum_{j=1}^{p} \beta_j \mathbb{E}[X_j] \right)
$$

$$
\sum_{j=1}^{p} \phi_j(\hat{f})
=  \hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{X})]
$$

This is the predicted value for the data point **x** minus the average predicted value.  Feature contributions can be negative.

----
---

### **Shapley values in blackbox models**  
In blackbox models we do not have such coefficients. We calculate Shapley value as the average marginal contribution of a feature value across all possible coalitions. See Salary Prediction example below.

---
**Shapley value Properties**  
The Shapley value is the only attribution method that satisfies the properties Efficiency, Symmetry, Dummy, and Additivity, which together can be considered a definition of a fair payout.
* **Efficiency** The feature contributions must add up to the difference of prediction for $x$ and the average.
* **Symmetry** The contributions of two feature values should be the same if they contribute equally to all possible coalitions.
* **Dummy** A feature that does not change the predicted value – regardless of which coalition of feature values it is added to – should have a Shapley value of 0
* **Additivity** The sum of the Shapley values for two combined games is the same as the sum of the individual games' Shapley values. This is useful in random forests (where the prediction is an average of many decision trees). The Additivity property guarantees that for a feature value, you can calculate the Shapley value for each tree individually, average them, and get the Shapley value for the feature value for the random forest.

---
**Methods to estimate Shapley values**  
All possible coalitions (sets) of feature values have to be evaluated with and without the j-th feature to calculate the exact Shapley value, which can be computationally challenging as the number of features increase, a MOnte Carlo approximation was proposed, to randomly select a limited (user-input number) of coalitions. See further down.

---
Strengths:
* The difference between the prediction and the average prediction is "fairly distributed" among the feature values of the instance; this arises from the Efficiency property of Shapley values that ensures the sum of the individual values for all players equals the total value of the cooperative game (i.e., here the Shapley values will sum up to the total outcome). It thus provides a full explanation of all features.
* Allows for contrastive explanation: instead of comparing a prediction to a average prediction of the entire dataset (as discussed above) we could compare it to the average of subset of data or a specific data; this tyoe of contrastiveness is lacking in LIME and Anchor
* Grounded in solid theory see refernce source for details: https://christophm.github.io/interpretable-ml-book/shapley.html (LIME assume lienarity in local neighborhood but no theory on why that would work)

Limitations"
* Returns a simple value per feature, but no prediction model like LIME or Anchor
* COmputationally expensive 2^k for each feature for each comparative analysis
* Shapley value explanations are not to be interpreted as local in the sense of gradients or neighborhood. (Bilodeau et al. 2024) For example, a positive Shapley value doesn’t mean that increasing the feature would increase the prediction. Instead, the Shapley value has to be interpreted with respect to the reference dataset that was used for the estimation.
* Like many other permutation-based interpretation methods, the Shapley value method suffers from inclusion of unrealistic data instances when features are correlated.

---
---


## IIe.i)  Example: Shapley Value Calculation for Salary Prediction (response variable is continuous)
Imagine a simple model that predicts an engineer's salary based on 3 binary features:
1. Experience (Exp)
2. Advanced Degree (Deg)
3. Manager Role (Mgr)

Suppose a specific employee has a prediction of $\$$150K, and they have the following feature values (Exp=1, Deg=1, Mgr=1). Suppose the baseline (average salary of all people in the dataset) is $100k. To determine how much did each feature (Exp, Deg, Mgr) contribute to this payout of 150K-100K= 50K, we will calculate the Shapely value of each feature for this specific person.

The table below shows what the model predicts for this person when different combinations of their features are hidden (masked) vs. active:

**Calculate Shapley values for a person with feature values (Exp=1, Deg=1, Mgr=1)**

**Suppose model predictions${^ {**}}$ for all feature combinations are as follows**

| Feature Combination | Model Prediction (\$) |
|---------------------|-----------------:|
| None (Base value)   | 100k |
| Exp only            | 120k |
| Deg only            | 105k |
| Mgr only            | 110k |
| Exp, Deg          | 131k |
| Exp, Mgr          | 134k |
| Deg, Mgr          | 120k |
| Exp, Deg, Mgr     | 150k |
---
  **Note: When computing the prediction for say "{Exp} alone", the model cannot simply leave Deg and Mgr blank because machine learning models require a value for every single feature input. Instead of plugging in a literal numeric 0, the algorithm simulates what happens when Deg and Mgr are "missing" or hidden (0 may make sense here but not in all cases). How this is done depends on the type of SHAP explainer:
  1. Conditional Expectation: The algorithm looks at the training dataset and calculates the average prediction of all rows where Exp = 1.It ignores individual values for Deg and Mgr by averaging them out across all possible variations found in your data. This is used in Tree-SHAP which is used for tree-based machine learning models such as Random forests and XGBoost.
  2. Reference Sampling (KernelSHAP)
  In model-agnostic explainer, you typically provide a "background dataset" (often the median or mean values of your data). If your background dataset sets the default state of Deg and Mgr to 0, then the model evaluates the specific combination for "{Exp} alone": Exp = 1, Deg = 0, Mgr = 0

---

As an example, lets calculate the Shapley Value for Exp. To do so we consider all possible coalitions involving (every possible subset of the other features): **Deg** and **Mgr**.

| Subset Without {Exp} | Prediction Without {Exp} | Prediction With {Exp} | Marginal Contribution | Game Theory Weight$^*$ | Weighted Contribution |
|--------------------|-----------------------:|--------------------:|----------------------:|-------------------:|----------------------:|
| Null set          | 100k | 120k | 120k - 100k = +20k | 1/3 | +6.67k |
| {Deg}              | 105k | 131k | 131k - 105k = +26k | 1/6 | +4.33k |
| {Mgr}              | 110k | 134k | 134k - 110k = +24k | 1/6 | +4.00k |
| {Deg, Mgr}         | 120k | 150k | 150k - 120k = +30k | 1/3 | +10.00k |

---

$$\phi_{Exp}=6.67 + 4.33 + 4.00 + 10.00$$

$$\boxed{\phi_{Exp} = +25k}$$

**Interpretation: Experience (Exp) contributes +$25k to the final model prediction.**

Note that: for abve example, the Shapley values for **Exp, Deg, and Mgr** must sum to 50K:

$$
\boxed{\phi_{Exp} + \phi_{Deg} + \phi_{Mgr}  = \text{sample prediction} - \text{baseline prediction} =150K-100K= \$50k}
$$

---
  *Note: The weights are theoretically coming from the game theory equation
  $$\phi(j)​=\sum_{S⊆N∖{j}} \frac{​∣S∣!(M−∣S∣−1)!}{M!}[v(S∪{j})−v(S)]$$
  $M$ is number of features, so the denomimator here is $3!=6$.
  - To understand the $1/3$ for the two scenarios, look back on the orginal Shapley coalition example. $Exp$ being first in the coalition can happen two ways, and $Exp$ being last in the colation can happen two ways.



Finally going to modern day SHAP as used in machine learning for explainability. NOte that the model is just a rewritten version of the additive model discussed above. Exhaustively calculating Shapely value for every sample is not computationally tractable. What the new work adds (modern day SHAP focus on) are efficient ways of calculating Shapely values for different type of features.
### **[SHAP](https://github.com/shap/shap#deep-learning-example-with-gradientexplainer-tensorflowkeraspytorch-models) (SHapley Additive exPlanations)**

SHAP assigns each feature an importance value for a particular prediction

Paper: [SHAP (SHapley Additive exPlanation)](https://papers.nips.cc/paper_files/paper/2017/file/8a20a8621978632d76c43dfd28b67767-Paper.pdf)

Computational: [SHAP_documentation](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html)

SHAP is represented as an additive feature attribution method (a linear model), connecting LIME and Shapley values. Specifcally, SHAP defines an explanation model as a linear function of binary variables representing the presence (1)) or absence (0)) of features. The model is defined as:
$$g(z^{\prime })=\phi _{0}+\sum _{j=1}^{M}\phi _{j}z_{j}^{\prime }$$ Where:
* $g$ is the explanation model.
* $z^{\prime }=(z_1',z_2',...., z_M')^T \in \{0,1\}^{M}$ is the coalition vector.
* $M$ is the number of features, the maximum coalition size.
* $\phi _{j}\in \mathbb{R}$ is the Shapley value (SHAP value) for feature $j$.
* $\phi _{0}=\mathbb{E}[f(X)]$ is the base value, or the average model output over the training dataset.

The above equation represents the idea that the model's output can be expressed as the sum of a baseline value and the contribution of each feature.

**Note**: features can be different than features of the model, e.g., in image data, the images are not represented on the pixel level, but aggregated into superpixels.

**Note**: The properties of Shapley values apply here too:
* Local accuracy or efficiency: For a specific input instance $x$, the SHAP values add up to the difference between the model's prediction for that instance, $f(x)$ and the base value, $\phi _{0}$, i.e., by setting colatition vector as vectors of 1's we get
$$\sum _{j=1}^{M}\phi _{j}=f(x)-\phi _{0}$$
* Satisfies properties of symmetry, dummy, and efficiency.

---
**SHAP estimation methods**  
Calculating the exact Shapley value is computationally intensive, as it requires evaluating $2^{M}$ coalitions for each instance. KernelSHAP, TreeSHAP, and Permutation methods are used. KernelSHAP help explain how SHAP connects Shapley values with LIME but are not computationally efficient and not commonly used. Permutation method is more common. TreeSHAP is specific to tree based methods: ML, e.g., deecisions trees, random forests, and gradient boosted trees. We will skip TreeSHAP.

---
**KernelSHAP**:   
Works by creating a simplified linear regression model to explain the model's predictions. It samples a subset of coalitions to train the linear model instead of evaluating all possible coalitions. Note this is similar to LIME concept, except that it uses a SHAP kernel for the proximity measure.

**Steps of KernelSHAP:**
* Sample coalition vectors $z_k'\in {0,1}^M, k\in{1,..., K} $ (1 = feature present in coalition, 0 = feature absent).$M$ is number of features. $K$ is number of coalitions to sample.
* Get prediction for each $z_k'$ by first converting it to the **original feature space** $h_x(z_k')$ and then applying the prediction model $\hat{f}:\hat{f}(h_x(z_k'))$.
* Compute the weight for each coalition with the **SHAP kernel weight function**.
* Find the **Shapley values** $\phi_k$, by solving for the coefficients of a weighted linear regression model that minimizes the **loss function**.

**SHAP kernel's weight function:**
:$$\pi _{x^{\prime }}(z^{\prime })=\frac{(M-1)}{(M \text{ choose } z') |z^{\prime }|(M-|z^{\prime }|)}$$
* $\pi _{x^{\prime }}(z^{\prime })$ is the weight for the coalition vector $z^{\prime }$.
* $M$  is the number of features.
* $|z^{\prime }|$ is the number of non-zero elements in $z^{\prime }$ (the size of the coalition).

*SHAP kernel weights the sampled coalitions to give more importance to coalitions with very few or almost all features*

**Loss function:**

$$Loss(f,g,\pi _{x^{\prime }})=\sum _{z^{\prime }\in Z}[f(h_{x}(z^{\prime }))-g(z^{\prime })]^{2}\pi _{x^{\prime }}(z^{\prime })$$
* $Z$ is the training data.
* $g$ is the weighted linear regression model $$g(z^{\prime })=\phi _{0}+\sum _{j=1}^{M}\phi _{j}z_{j}^{\prime }$$
* $h_x$ for tabular data: maps a coalition to a valid instance; for present features (1), it maps to the feature values of $x$. For absent features (0), it maps to the values of a randomly sampled data instance. [See Figure 18.1](https://christophm.github.io/interpretable-ml-book/shap.html#fig-shap-simplified-feature)
*  $h_x$ for image data: maps coalitions of superpixels (sp) to images. Superpixels are groups of pixels. For present features (1), $h_x$ returns the corresponding part of the original image. For absent features (0), $h_x$ greys out the corresponding area.  [See Figure 18.2](https://christophm.github.io/interpretable-ml-book/shap.html#fig-shap-images). Assigning the average color of surrounding pixels or similar would also be an option.
* The resulting coefficients $\phi _{j}$ of this linear model are the approximate SHAP values

**Note**: The big difference between SHAP and LIME is the weighting of the instances in the regression model. LIME weights the instances according to how close they are to the original instance. SHAP weights the sampled instances according to the weight the coalition would get in the Shapley value estimation. Small coalitions (few 1’s) and large coalitions (i.e. many 1’s) get the largest weights. The intuition behind it is: We learn most about individual features if we can study their effects in isolation. If a coalition consists of a single feature, we can learn about this feature’s isolated main effect on the prediction. If a coalition consists of all but one feature, we can learn about this feature’s total effect (main effect plus feature interactions). If a coalition consists of half the features, we learn little about an individual feature’s contribution, as there are many possible coalitions with half of the features.


SHAP computtaional tools
* SHAP webbrowser-based vizualization tool {SHAP tool- Poloclub](https://poloclub.github.io/webshap/?model=image)



**Local v Global explanations using SHAP**

* Local explanation: What we discussed thus far is SHAP for an individual instance/sample, i.e., it only explains feature contributions speicifc to that instance to the payout (predicted instance - average prediction)
* Global explanations: Shapley values can be combined into global explanations. If we run SHAP for every instance, we get a matrix of Shapley values (one row per data instance and one column per feature). We can interpret the entire model by analyzing the Shapley values in this matrix.

---

### SHAP for DeepLearning models
While KernelSHAP is model agnostic, it has computational limits. DeepExplainer and GradientExplainer are model-specific (deep learning specific) approximations designed to overcome those computational limits of KernelSHAP.
KernelSHAP treats the model as a completely black box,and obtains Shapely value through the Brute force noted above. DeepExplainer and GradientExplainer leverage the the model's architecture, such as the propagation of gradients that are part of the algorithms, to calculate Shapley values significantly faster.

---

## IIe. ii) SHAP for multiclass classification

Deep learning setup for classification problem is different than in other machine learning such as XGBoost. In XGBoost it trains C trees, where C is number of classes, using a one-versus-rest approach, e.g., tree for class 1 is classifies CLass 1 against all other classes, etc. It then internally goes through all models to calculate the logis scores for each class.  

In deep learning, most times, we setop the output layer itself as number of classes, and are assume what we learn are the logit scores. SHAP values are calculated using these logit-scores. For every sample, our output is a vector $[c1_{logit}, c2_{logit},...]$. For each sample, SHAP values are calculated for each class, by looking at what made the model give that sample that logit comapred to basline logit.

For example, suppose we are classifying blacklegged_tick_status in the U.S.,
- Cass labels:  'no reports', 'reported', 'established'.
- Independent Features: annual temperature, annual precipitation etc.,
- Amherst (one sample) gets a logit score of say $[2, 0.2, 5]$,  
- Baseline logit score (average across all samples - counties in the US) is $[0.5, 0.4, 0.3]$.
- For Amherst, the payout for class 1 is $2-0.5$, payout for class 2 is $0.2-0.4$, and payout for class 3 is $5-0.3$.
- SHAP value helps explain how much did Amherst' temperature, annual precipitation contribute to this payout (calculates it for each class).

Thus, the SHAP plots for classification problems presented for each class. See code example corresponding to above.

### Commonly used plots for SHAP
- Absolute Mean SHAP values: for each feature, takes average of absolute values across all samples
- Beeswarm: Looks at Shapley values of each sample for each feature
- SHAP dependence plot